### 필수 라이브러리 불러오기

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import json
import os

### 데이터 불러오기 및 병합 (현재는 더미 데이터로 세팅)

In [ ]:
# 팀원들이 데이터를 주기 전까지 테스트할 가짜(Dummy) 데이터
dummy_data = {
    '자치구': ['강남구', '관악구', '종로구', '노원구', '마포구'],
    '범죄율': [3.5, 4.2, 2.1, 1.8, 2.9],      # 부정 지표
    'CCTV': [1200, 850, 600, 950, 1100],     # 긍정 지표
    '파출소': [15, 12, 8, 10, 11],            # 긍정 지표
    '안심귀갓길': [40, 35, 20, 25, 30],       # 긍정 지표
    '가로등': [3500, 2800, 1500, 2100, 3100]  # 긍정 지표
}

# 데이터프레임으로 변환
df = pd.DataFrame(dummy_data)
display(df) # 데이터가 잘 들어갔는지 확인

### 데이터 정규화 및 치안안전지수 계산 (가중치 50:30:20 적용)

In [ ]:
# 1. 스케일러 준비
scaler = MinMaxScaler()

# 2. 긍정 지표 정규화 (0~100점 변환)
positive_cols = ['CCTV', '파출소', '안심귀갓길', '가로등']
df[positive_cols] = scaler.fit_transform(df[positive_cols]) * 100

# 3. 부정 지표(범죄율) 정규화 후 뒤집기 (낮을수록 점수가 높게)
df['범죄율'] = (1 - scaler.fit_transform(df[['범죄율']])) * 100

# 4. 가중치를 반영한 최종 안전지수 계산
df['치안안전지수'] = (
    df['범죄율'] * 0.50 +
    df['CCTV'] * 0.15 +
    df['파출소'] * 0.15 +
    df['안심귀갓길'] * 0.10 +
    df['가로등'] * 0.10
)

# 소수점 둘째 자리까지 반올림 후 점수가 높은 순으로 정렬
df['치안안전지수'] = df['치안안전지수'].round(2)
df_final = df.sort_values('치안안전지수', ascending=False).reset_index(drop=True)

display(df_final[['자치구', '치안안전지수']])

### 프론트엔드 전달용 JSON 파일 추출

In [ ]:
# data/processed 폴더가 없으면 생성
os.makedirs('../data/processed', exist_ok=True)

# 프론트엔드가 쓰기 좋은 JSON 형태로 저장
df_final.to_json('../data/processed/seoul_safety_index.json', orient='records', force_ascii=False)

print()